# `prepare_alm` の動作確認

HEALPix/healpy形式で $m\geq0$ だけを保持した $(T,E,B)$ 係数を、Convolution内部で使う $(0,+2,-2)$ と $m=-l,\ldots,l$ の形へ変換します。

In [ ]:
using Scanvolver
using Test

## 1. 小さい入力を作る

入力の各行は $(T,E,B)$、列はHEALPix/healpyのpacked alm indexです。ここでは $l=2,m=1$ の係数だけを非ゼロにします。

In [ ]:
cp = ConvolutionParams(
    nside_output=2,
    lstart=0,
    lstop=3,
    lmax=3,
    numofsky=1,
)

packed_index(l, m, lmax) =
    m * (lmax + 1) - m * (m - 1) ÷ 2 + (l - m) + 1

coefficient_count = (cp.lmax + 1) * (cp.lmax + 2) ÷ 2
alm_teb = zeros(ComplexF64, 3, coefficient_count)
l, m = 2, 1
index = packed_index(l, m, cp.lmax)
alm_teb[:, index] .= ComplexF64[1 + 2im, 3 - 0.5im, -2 + 4im]
alm_teb

## 2. `prepare_alm` を呼ぶ

既定の `input_basis=:teb` が $(T,E,B)$ を $(0,+2,-2)$ へ変換します。出力は $l$ ごとのragged sliceなので、不要なpaddingをCPU上に保持しません。

In [ ]:
prepared = prepare_alm(cp, alm_teb)
slice_l2 = alm_slice(prepared, 2)
(
    prepared=prepared,
    spin_order=(0, +2, -2),
    m_order=collect(-2:2),
    block_sizes=size.(slice_l2),
)

In [ ]:
spin0, spin_plus2, spin_minus2 = slice_l2
rows = [
    (m=m_value, spin0=spin0[1, m_value + l + 1],
     spin_plus2=spin_plus2[1, m_value + l + 1],
     spin_minus2=spin_minus2[1, m_value + l + 1])
    for m_value in -l:l
]

## 3. 変換式と負の $m$ を確認する

$a_{+2}=-(a_E+i a_B)$、$a_{-2}=-(a_E-i a_B)$ と、$a_s(l,-m)=(-1)^m\,a_{-s}(l,m)^*$ を直接チェックします。

In [ ]:
positive = m + l + 1
negative = -m + l + 1
expected_plus2 = -(alm_teb[2, index] + im * alm_teb[3, index])
expected_minus2 = -(alm_teb[2, index] - im * alm_teb[3, index])

@test spin0[1, positive] == alm_teb[1, index]
@test spin_plus2[1, positive] == expected_plus2
@test spin_minus2[1, positive] == expected_minus2
@test spin0[1, negative] == -conj(spin0[1, positive])
@test spin_plus2[1, negative] == -conj(spin_minus2[1, positive])
@test spin_minus2[1, negative] == -conj(spin_plus2[1, positive])

"all six checks passed"

## 4. すでにspin基底の場合

入力がすでに $(0,+2,-2)$ なら `input_basis=:spin` を指定します。

In [ ]:
alm_spin = similar(alm_teb)
alm_spin[1, :] .= alm_teb[1, :]
alm_spin[2, :] .= -(alm_teb[2, :] .+ im .* alm_teb[3, :])
alm_spin[3, :] .= -(alm_teb[2, :] .- im .* alm_teb[3, :])
prepared_spin = prepare_alm(cp, alm_spin; input_basis=:spin)

maximum_difference = maximum(
    maximum(abs.(a .- b))
    for l_value in cp.lstart:cp.lstop
    for (a, b) in zip(alm_slice(prepared, l_value), alm_slice(prepared_spin, l_value))
)
@test maximum_difference == 0
maximum_difference

## 5. 複数skyとFloat32

複数skyは行列のvectorとして渡します。GPU計算用の精度に合わせて `precision=Float32` も選べます。

In [ ]:
cp_two_skies = ConvolutionParams(cp; numofsky=2)
prepared_two = prepare_alm(
    cp_two_skies,
    [alm_teb, 2 .* alm_teb];
    precision=Float32,
)

(
    prepared=prepared_two,
    element_type=eltype(prepared_two),
    l2_spin0_size=size(alm_slice(prepared_two, 2)[1]),
)